# 1. AOI Selection — Google Earth Engine

Draw an Area of Interest (AOI) on an interactive map and export it as **GeoJSON**, **KML**, and **KMZ**.

Run the cells top to bottom:
1. Authenticate / initialize Earth Engine.
2. Display the map and draw your AOI (rectangle or polygon tool, one shape).
3. Run the export cell to write the files.
4. Verify the export.

The output files (named `AOI_NAME` below) are what `2_Sentinel_Search.ipynb` reads — run this notebook first.

## Setup — authenticate and initialize Earth Engine

Set `EE_PROJECT` to a Google Cloud project that has the Earth Engine API enabled
(see https://console.cloud.google.com — any project you've registered at
https://code.earthengine.google.com/register works). `ee.Authenticate()` only
needs to run once per machine; after that it's cached and the line is a no-op.

In [1]:
import ee
import geemap

EE_PROJECT = "rosy-precinct-498822-e1"  # <-- your GEE-enabled Cloud project

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")

Earth Engine initialized.


## Draw the AOI

Use the rectangle or polygon tool on the left toolbar of the map below to draw
**one** shape over your area of interest. Draw only one shape — if you draw
more than one, the export step below uses the last one drawn.

In [2]:
Map = geemap.Map(center=[20, 0], zoom=3)
Map.add_basemap("Hybrid")
Map.add_basemap("Roadmap")
Map

Map(center=[20, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', trans…

## Export the drawn AOI

Set `AOI_NAME` and `OUTPUT_DIR`, then run the cell. It reads the last shape
drawn on the map above (`Map.user_roi`) and writes `.geojson`, `.kml`, and
`.kmz` files. `2_Sentinel_Search.ipynb` looks for these same `AOI_NAME` /
`OUTPUT_DIR` values, so keep them noted if you change them.

In [4]:
from aoi_export import export_all

AOI_NAME = "my_aoi"
OUTPUT_DIR = "output"

geometry = Map.user_roi
if geometry is None:
    raise ValueError("No shape drawn yet — draw a rectangle or polygon on the map above, then re-run this cell.")

paths = export_all(geometry, OUTPUT_DIR, name=AOI_NAME)
for fmt, path in paths.items():
    print(f"{fmt}: {path.resolve()}")

geojson: C:\Users\Say70\OneDrive - Mississippi State University\Desktop\Satellite Data Search\output\my_aoi.geojson
kml: C:\Users\Say70\OneDrive - Mississippi State University\Desktop\Satellite Data Search\output\my_aoi.kml
kmz: C:\Users\Say70\OneDrive - Mississippi State University\Desktop\Satellite Data Search\output\my_aoi.kmz


## Verify the exported AOI

Load the GeoJSON back and overlay it on a fresh map as a sanity check.

In [5]:
check_map = geemap.Map()
check_map.add_geojson(str(paths["geojson"]), layer_name=AOI_NAME)
check_map.centerObject(geometry, zoom=10)
check_map

Map(center=[33.47402395264986, -88.77396550000127], controls=(WidgetControl(options=['position', 'transparent_…